# WiLI-2018 Language Detection – Clean Benchmark & Model Export
This notebook trains and saves:
- Classical ML: Complement NB, SGD, Passive Aggressive, Ridge, Linear SVC, Nearest Centroid
- LangDetect‑style Naïve Bayes
- fastText, GlotLID, CLD3 (PyTorch)
- High‑capacity Character‑level CNN

In [ ]:
import os
import time
import pickle
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import SGDClassifier, RidgeClassifier, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import NearestCentroid

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

# Fix memory fragmentation on Kaggle
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Device & GPU count
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE} | GPUs: {NUM_GPUS}")

# Directory to save all models
SAVE_DIR = "./backend/weights/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to {SAVE_DIR}")

In [ ]:
DATA_DIR = "./dataset/raw/"

with open(DATA_DIR + "x_train.txt", encoding="utf-8") as f:
    X_train_raw = [l.strip() for l in f]
with open(DATA_DIR + "y_train.txt", encoding="utf-8") as f:
    y_train_raw = [l.strip() for l in f]
with open(DATA_DIR + "x_test.txt", encoding="utf-8") as f:
    X_test_raw = [l.strip() for l in f]
with open(DATA_DIR + "y_test.txt", encoding="utf-8") as f:
    y_test_raw = [l.strip() for l in f]

# Drop empty lines
def clean(texts, labels):
    pairs = [(t, l) for t, l in zip(texts, labels) if t.strip()]
    return zip(*pairs)

X_train_raw, y_train_raw = clean(X_train_raw, y_train_raw)
X_test_raw,  y_test_raw  = clean(X_test_raw,  y_test_raw)
X_train_raw, y_train_raw = list(X_train_raw), list(y_train_raw)
X_test_raw,  y_test_raw  = list(X_test_raw),  list(y_test_raw)

# Label encode (fit on all labels to cover test set)
le = LabelEncoder()
le.fit(y_train_raw + y_test_raw)
y_train_enc = le.transform(y_train_raw)
y_test_enc  = le.transform(y_test_raw)

NUM_CLASSES = len(le.classes_)
print(f"Train samples: {len(X_train_raw):,}  |  Test samples: {len(X_test_raw):,}")
print(f"Languages: {NUM_CLASSES}")

# Save label encoder – crucial for inference
with open(SAVE_DIR + "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("LabelEncoder saved.")

In [ ]:
print("CLASSICAL ML MODELS (char_wb 2-4 grams, 50k features)")

vec_clf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=50_000,
    sublinear_tf=True
)

X_train_tfidf = vec_clf.fit_transform(X_train_raw)
X_test_tfidf  = vec_clf.transform(X_test_raw)
print(f"TF-IDF shape: {X_train_tfidf.shape}")

# Save vectorizer
joblib.dump(vec_clf, SAVE_DIR + "vectorizer_char_wb_2_4.pkl")

models_clf = {
    "ComplementNB": ComplementNB(),
    "SGDClassifier": SGDClassifier(loss="modified_huber", class_weight="balanced",
                                   random_state=42, n_jobs=-1),
    "PassiveAggressive": PassiveAggressiveClassifier(class_weight="balanced",
                                                     random_state=42, n_jobs=-1),
    "RidgeClassifier": RidgeClassifier(class_weight="balanced"),
    "LinearSVC": LinearSVC(class_weight="balanced", random_state=42, max_iter=2000),
    "NearestCentroid": NearestCentroid(),
}

results_clf = []
for name, model in models_clf.items():
    print(f"\n--- {name} ---")
    t0 = time.time()
    model.fit(X_train_tfidf, y_train_enc)
    train_time = time.time() - t0
    y_pred = model.predict(X_test_tfidf)
    macro_f1 = f1_score(y_test_enc, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_test_enc, y_pred)
    print(f"  Accuracy: {acc:.4f}  |  Macro F1: {macro_f1:.4f}  |  Train: {train_time:.1f}s")
    results_clf.append({"Model": name, "Accuracy": acc, "Macro F1": macro_f1})
    joblib.dump(model, SAVE_DIR + f"clf_{name}.pkl")

print("\nClassical models saved.")
pd.DataFrame(results_clf).sort_values("Macro F1", ascending=False)

In [ ]:
print("LANGDETECT‑STYLE COMPLEMENT NAIVE BAYES (char_wb 1-3 grams)")

vec_ld = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(1, 3),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents=None
)

X_train_ld = vec_ld.fit_transform(X_train_raw)
X_test_ld  = vec_ld.transform(X_test_raw)
print(f"TF-IDF shape: {X_train_ld.shape}")

model_ld = ComplementNB(alpha=0.1)
t0 = time.time()
model_ld.fit(X_train_ld, y_train_enc)
train_time = time.time() - t0

y_pred_ld = model_ld.predict(X_test_ld)
macro_f1_ld = f1_score(y_test_enc, y_pred_ld, average="macro", zero_division=0)
acc_ld = accuracy_score(y_test_enc, y_pred_ld)
print(f"  Accuracy: {acc_ld:.4f}  |  Macro F1: {macro_f1_ld:.4f}  |  Train: {train_time:.1f}s")

# Save vectorizer and model
joblib.dump(vec_ld, SAVE_DIR + "vectorizer_char_wb_1_3_langdetect.pkl")
joblib.dump(model_ld, SAVE_DIR + "langdetect_style_complement_nb.pkl")
print("LangDetect‑style model saved.")

In [ ]:
print("DEEP LEARNING MODELS (fastText / GlotLID / CLD3)")

# ---------- Shared utilities ----------
def char_ngrams(text, min_n=2, max_n=4):
    text = f" {text} "
    ngrams = []
    for n in range(min_n, max_n + 1):
        ngrams += [text[i:i+n] for i in range(len(text) - n + 1)]
    return ngrams

def hash_ngram(ngram, bucket_size):
    h = 2166136261
    for ch in ngram.encode("utf-8", errors="replace"):
        h ^= ch
        h = (h * 16777619) & 0xFFFFFFFF
    return (h % (bucket_size - 1)) + 1

class LangDataset(Dataset):
    def __init__(self, texts, labels, bucket_size, min_n, max_n):
        self.texts = texts
        self.labels = labels
        self.bucket_size = bucket_size
        self.min_n = min_n
        self.max_n = max_n

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ngrams = char_ngrams(self.texts[idx], self.min_n, self.max_n)
        ids = [hash_ngram(g, self.bucket_size) for g in ngrams] if ngrams else [1]
        return torch.tensor(ids, dtype=torch.long), self.labels[idx]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

def train_eval(model, train_loader, test_loader, epochs=5, lr=1e-3, is_cld3=False, accum_steps=2):
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler('cuda')

    train_start = time.time()
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=True)
        for i, batch in enumerate(train_pbar):
            if is_cld3:
                uni, bi, tri, labels = [x.to(DEVICE, non_blocking=True) for x in batch]
                with autocast('cuda'):
                    outputs = model(uni, bi, tri)
                    loss = criterion(outputs, labels) / accum_steps
            else:
                seqs, lengths, labels = [x.to(DEVICE, non_blocking=True) for x in batch]
                with autocast('cuda'):
                    outputs = model(seqs, targets=labels)
                    raw_loss = outputs if hasattr(model.module if NUM_GPUS>1 else model, 'is_fasttext') else criterion(outputs, labels)
                    loss = raw_loss / accum_steps

            if NUM_GPUS > 1 and loss.dim() > 0:
                loss = loss.mean()

            scaler.scale(loss).backward()
            if (i+1) % accum_steps == 0 or (i+1) == len(train_loader):
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
            train_pbar.set_postfix(loss=f"{(loss.item() * accum_steps):.4f}")
        scheduler.step()
        torch.cuda.empty_cache()

    train_time = time.time() - train_start

    model.eval()
    all_preds, all_labels = [], []
    inf_start = time.time()
    with torch.no_grad():
        eval_pbar = tqdm(test_loader, desc="Evaluating", leave=True)
        for batch in eval_pbar:
            if is_cld3:
                uni, bi, tri, labels = [x.to(DEVICE, non_blocking=True) for x in batch]
                with autocast('cuda'):
                    preds = model(uni, bi, tri).argmax(dim=1)
            else:
                seqs, lengths, labels = [x.to(DEVICE, non_blocking=True) for x in batch]
                with autocast('cuda'):
                    preds = model(seqs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    inf_time = time.time() - inf_start

    return np.array(all_preds), np.array(all_labels), train_time, inf_time

BATCH_SIZE = 256
NUM_WORKERS = 2

# ---------- 5.1 fastText ----------
class FastTextModel(nn.Module):
    def __init__(self, bucket_size, embed_dim, num_classes):
        super().__init__()
        self.is_fasttext = True
        self.embedding = nn.EmbeddingBag(bucket_size, embed_dim, mode="mean", padding_idx=0, sparse=False)
        self.classifier = nn.AdaptiveLogSoftmaxWithLoss(
            embed_dim, num_classes, cutoffs=[num_classes//4, num_classes//2], div_value=2.0
        )

    def forward(self, x, targets=None):
        emb = self.embedding(x)
        if targets is not None:
            return self.classifier(emb, targets).loss
        return self.classifier.log_prob(emb)

BUCKET_FT = 2_000_000
EMBED_FT = 64

print("\n--- fastText ---")
ft_train = LangDataset(X_train_raw, y_train_enc, BUCKET_FT, min_n=2, max_n=4)
ft_test  = LangDataset(X_test_raw,  y_test_enc,  BUCKET_FT, min_n=2, max_n=4)
ft_train_loader = DataLoader(ft_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
ft_test_loader  = DataLoader(ft_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

ft_model = FastTextModel(BUCKET_FT, EMBED_FT, NUM_CLASSES)
ft_preds, ft_labels, ft_tt, ft_it = train_eval(ft_model, ft_train_loader, ft_test_loader, epochs=5, lr=0.1)
acc_ft = accuracy_score(ft_labels, ft_preds)
f1_ft  = f1_score(ft_labels, ft_preds, average="macro", zero_division=0)
print(f"  Accuracy: {acc_ft:.4f}  |  Macro F1: {f1_ft:.4f}  |  Train: {ft_tt:.1f}s")

torch.save(ft_model.state_dict(), SAVE_DIR + "fasttext_weights.pth")

# ---------- 5.2 GlotLID ----------
class GlotLIDModel(nn.Module):
    def __init__(self, bucket_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.EmbeddingBag(bucket_size, embed_dim, mode="mean", padding_idx=0, sparse=False)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, targets=None):
        return self.classifier(self.embedding(x))

GLOT_BUCKET = 2_000_000
GLOT_EMBED = 128

print("\n--- GlotLID ---")
gl_train = LangDataset(X_train_raw, y_train_enc, GLOT_BUCKET, min_n=2, max_n=5)
gl_test  = LangDataset(X_test_raw,  y_test_enc,  GLOT_BUCKET, min_n=2, max_n=5)
gl_train_loader = DataLoader(gl_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
gl_test_loader  = DataLoader(gl_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

gl_model = GlotLIDModel(GLOT_BUCKET, GLOT_EMBED, NUM_CLASSES)
gl_preds, gl_labels, gl_tt, gl_it = train_eval(gl_model, gl_train_loader, gl_test_loader, epochs=5, lr=0.1)
acc_gl = accuracy_score(gl_labels, gl_preds)
f1_gl  = f1_score(gl_labels, gl_preds, average="macro", zero_division=0)
print(f"  Accuracy: {acc_gl:.4f}  |  Macro F1: {f1_gl:.4f}  |  Train: {gl_tt:.1f}s")

torch.save(gl_model.state_dict(), SAVE_DIR + "glotlid_weights.pth")

# ---------- 5.3 CLD3 ----------
class CLD3Dataset(Dataset):
    def __init__(self, texts, labels, bucket_size):
        self.texts = texts
        self.labels = labels
        self.bucket_size = bucket_size

    def __len__(self): return len(self.texts)

    def _get_ids(self, text, n):
        text = f" {text} "
        ngrams = [text[i:i+n] for i in range(len(text) - n + 1)]
        if not ngrams: return [1]
        return [hash_ngram(g, self.bucket_size) for g in ngrams]

    def __getitem__(self, idx):
        t = self.texts[idx]
        return (
            torch.tensor(self._get_ids(t, 1), dtype=torch.long),
            torch.tensor(self._get_ids(t, 2), dtype=torch.long),
            torch.tensor(self._get_ids(t, 3), dtype=torch.long),
            self.labels[idx]
        )

def cld3_collate(batch):
    uni, bi, tri, labels = zip(*batch)
    pad = lambda seqs: nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return pad(uni), pad(bi), pad(tri), torch.tensor(labels, dtype=torch.long)

class CLD3Model(nn.Module):
    def __init__(self, bucket_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.emb1 = nn.EmbeddingBag(bucket_size, embed_dim, mode="mean", padding_idx=0, sparse=False)
        self.emb2 = nn.EmbeddingBag(bucket_size, embed_dim, mode="mean", padding_idx=0, sparse=False)
        self.emb3 = nn.EmbeddingBag(bucket_size, embed_dim, mode="mean", padding_idx=0, sparse=False)
        self.hidden = nn.Linear(embed_dim * 3, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, uni, bi, tri):
        x = torch.cat([self.emb1(uni), self.emb2(bi), self.emb3(tri)], dim=1)
        x = self.dropout(self.relu(self.hidden(x)))
        return self.classifier(x)

CLD3_BUCKET = 1_000_000
CLD3_EMBED = 64
CLD3_HIDDEN = 256

print("\n--- CLD3 ---")
cld3_train = CLD3Dataset(X_train_raw, y_train_enc, CLD3_BUCKET)
cld3_test  = CLD3Dataset(X_test_raw,  y_test_enc,  CLD3_BUCKET)
cld3_train_loader = DataLoader(cld3_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=cld3_collate, num_workers=NUM_WORKERS, pin_memory=True)
cld3_test_loader  = DataLoader(cld3_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=cld3_collate, num_workers=NUM_WORKERS, pin_memory=True)

cld3_model = CLD3Model(CLD3_BUCKET, CLD3_EMBED, CLD3_HIDDEN, NUM_CLASSES)
cld3_preds, cld3_labels, cld3_tt, cld3_it = train_eval(cld3_model, cld3_train_loader, cld3_test_loader, epochs=5, lr=0.05, is_cld3=True)
acc_cld3 = accuracy_score(cld3_labels, cld3_preds)
f1_cld3  = f1_score(cld3_labels, cld3_preds, average="macro", zero_division=0)
print(f"  Accuracy: {acc_cld3:.4f}  |  Macro F1: {f1_cld3:.4f}  |  Train: {cld3_tt:.1f}s")

torch.save(cld3_model.state_dict(), SAVE_DIR + "cld3_weights.pth")

print("\nAll DL models saved.")

# Cleanup
import gc
del ft_model, gl_model, cld3_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ==================== 6. Character‑level CNN (High Capacity) ====================
print("\nCHARACTER‑LEVEL CNN (High Capacity)")

class CharDataset(Dataset):
    def __init__(self, texts, labels, vocab_size, max_seq_len):
        self.texts = texts
        self.labels = labels
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = f" {self.texts[idx]} "[:self.max_seq_len]
        ids = [((hash(ch) % (self.vocab_size - 1)) + 1) for ch in text]
        return torch.tensor(ids, dtype=torch.long), self.labels[idx]

def char_collate_fn(batch):
    seqs, labels = zip(*batch)
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    if padded.size(1) < 7:
        pad_size = 7 - padded.size(1)
        padded = F.pad(padded, (0, pad_size), value=0)
    return padded, torch.tensor(labels, dtype=torch.long)

class CharCNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes, num_filters):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(num_filters) for _ in filter_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        conv_results = []
        for conv, bn in zip(self.convs, self.bns):
            c = F.relu(bn(conv(x)))
            pooled = torch.max(c, dim=2)[0]
            conv_results.append(pooled)
        out = torch.cat(conv_results, dim=1)
        out = self.dropout(out)
        return self.fc(out)

def train_eval_cnn(model, train_loader, test_loader, epochs=20, lr=1e-3, accum_steps=4):
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = GradScaler('cuda')

    train_start = time.time()
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=True)
        for i, (seqs, labels) in enumerate(train_pbar):
            seqs, labels = seqs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with autocast('cuda'):
                outputs = model(seqs)
                loss = criterion(outputs, labels) / accum_steps
            if NUM_GPUS > 1 and loss.dim() > 0:
                loss = loss.mean()
            scaler.scale(loss).backward()
            if (i+1) % accum_steps == 0 or (i+1) == len(train_loader):
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
            train_pbar.set_postfix(loss=f"{(loss.item() * accum_steps):.4f}", lr=f"{scheduler.get_last_lr()[0]:.5f}")
        scheduler.step()
        torch.cuda.empty_cache()

    train_time = time.time() - train_start

    model.eval()
    all_preds, all_labels = [], []
    inf_start = time.time()
    with torch.no_grad():
        eval_pbar = tqdm(test_loader, desc="Evaluating", leave=True)
        for seqs, labels in eval_pbar:
            seqs = seqs.to(DEVICE, non_blocking=True)
            with autocast('cuda'):
                preds = model(seqs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    inf_time = time.time() - inf_start

    return np.array(all_preds), np.array(all_labels), train_time, inf_time

# Hyperparameters
CHAR_VOCAB = 65536
CHAR_EMBED = 128
NUM_FILTERS = 256
FILTER_SIZES = [2, 3, 4, 5, 6, 7]
MAX_SEQ_LEN = 768
BATCH_SIZE_CNN = 128
ACCUM_STEPS = 4
EPOCHS = 20
NUM_WORKERS = 2

print(f"  Batch size: {BATCH_SIZE_CNN}  |  Accum steps: {ACCUM_STEPS}  |  Epochs: {EPOCHS}")

cnn_train = CharDataset(X_train_raw, y_train_enc, CHAR_VOCAB, max_seq_len=MAX_SEQ_LEN)
cnn_test  = CharDataset(X_test_raw,  y_test_enc,  CHAR_VOCAB, max_seq_len=MAX_SEQ_LEN)
cnn_train_loader = DataLoader(cnn_train, batch_size=BATCH_SIZE_CNN, shuffle=True,
                              collate_fn=char_collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
cnn_test_loader  = DataLoader(cnn_test,  batch_size=BATCH_SIZE_CNN, shuffle=False,
                              collate_fn=char_collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

cnn_model = CharCNNModel(vocab_size=CHAR_VOCAB, embed_dim=CHAR_EMBED,
                         num_classes=NUM_CLASSES, filter_sizes=FILTER_SIZES, num_filters=NUM_FILTERS)

cnn_preds, cnn_labels, cnn_tt, cnn_it = train_eval_cnn(
    cnn_model, cnn_train_loader, cnn_test_loader, epochs=EPOCHS, lr=1e-3, accum_steps=ACCUM_STEPS
)

acc_cnn = accuracy_score(cnn_labels, cnn_preds)
f1_cnn  = f1_score(cnn_labels, cnn_preds, average="macro", zero_division=0)
print(f"\nCharCNN-HighCap  Accuracy: {acc_cnn:.4f}  |  Macro F1: {f1_cnn:.4f}  |  Train: {cnn_tt:.1f}s")

torch.save(cnn_model.state_dict(), SAVE_DIR + "charcnn_highcap_weights.pth")

# Cleanup
del cnn_model, cnn_train_loader, cnn_test_loader
gc.collect()
torch.cuda.empty_cache()

print("\nCharCNN saved.")

In [ ]:

print("ALL MODELS TRAINED AND SAVED")

print(f"Directory: {SAVE_DIR}")
for f in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(SAVE_DIR + f) / 1e6
    print(f"  {f:<50} {sz:>6.1f} MB")
print("="*60)